In [1]:
# Project 3: AI Recommendation Logic - Tech Stack Recommender
# DecodeLabs - Industrial Training Kit
# Content-Based Filtering using TF-IDF & Cosine Similarity

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("="*60)
print("🎯 DecodeLabs - Project 3: AI Recommendation Logic")
print("📌 Tech Stack Recommender (Content-Based Filtering)")
print("="*60)

# ============================================
# STEP 1: CREATE THE DATASET (Tech Roles & Required Skills)
# ============================================

print("\n📊 STEP 1: Loading Tech Roles Dataset")
print("-"*40)

# Dataset: Job roles with their required skills (comma-separated)
data = {
    'role': [
        'Frontend Developer',
        'Backend Developer',
        'Full Stack Developer',
        'Data Scientist',
        'Machine Learning Engineer',
        'DevOps Engineer',
        'Cloud Architect',
        'Mobile Developer (iOS)',
        'Mobile Developer (Android)',
        'Cybersecurity Analyst',
        'Database Administrator',
        'QA Engineer',
        'Product Manager',
        'UI/UX Designer',
        'Game Developer'
    ],
    'skills': [
        'JavaScript, React, HTML, CSS, Vue, Angular, Frontend',
        'Python, Java, Node.js, SQL, REST APIs, Spring Boot, Django',
        'JavaScript, Python, React, Node.js, MongoDB, SQL, HTML, CSS',
        'Python, SQL, Statistics, Machine Learning, Pandas, NumPy, Data Visualization',
        'Python, TensorFlow, PyTorch, Scikit-learn, Deep Learning, NLP, Computer Vision',
        'Docker, Kubernetes, Jenkins, AWS, CI/CD, Linux, Terraform',
        'AWS, Azure, GCP, Cloud Computing, Terraform, Kubernetes, Networking',
        'Swift, iOS, Xcode, UIKit, Core Data, Mobile Development',
        'Kotlin, Java, Android, Android Studio, Mobile Development, REST APIs',
        'Network Security, Kali Linux, Cryptography, Risk Assessment, Penetration Testing',
        'SQL, PostgreSQL, MySQL, Database Design, Query Optimization, MongoDB',
        'Testing, Selenium, Automation, JUnit, pytest, CI/CD',
        'Agile, Scrum, Product Management, JIRA, Market Research, Leadership',
        'Figma, Adobe XD, Wireframing, Prototyping, User Research, HTML, CSS',
        'C#, Unity, C++, Game Physics, 3D Modeling, Unreal Engine'
    ]
}

df = pd.DataFrame(data)
print(f"   ✅ Loaded {len(df)} tech roles")
print(f"\n   📋 Available Roles:")
for i, role in enumerate(df['role'], 1):
    print(f"      {i}. {role}")

# ============================================
# STEP 2: TEXT PREPROCESSING (Convert skills to text documents)
# ============================================

print("\n📊 STEP 2: Preparing Skill Data for TF-IDF")
print("-"*40)

# The skills column is our "document" for TF-IDF
documents = df['skills'].tolist()

print("   Sample document (Backend Developer):")
print(f"   → {documents[1]}")
print("\n   📌 TF-IDF will:")
print("      • Penalize common words (like 'Python' if too frequent)")
print("      • Reward unique/specific skills")

# ============================================
# STEP 3: APPLY TF-IDF VECTORIZATION
# ============================================

print("\n📊 STEP 3: Applying TF-IDF Vectorization")
print("-"*40)

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',  # Remove common words
    token_pattern=r'[a-zA-Z]+(?: [a-zA-Z]+)*'  # Capture multi-word skills
)

# Transform documents into TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(documents)

print(f"   ✅ TF-IDF Matrix created!")
print(f"   • Shape: {tfidf_matrix.shape}")
print(f"   • {tfidf_matrix.shape[0]} documents (roles)")
print(f"   • {tfidf_matrix.shape[1]} unique skill features")

# Show some feature names
feature_names = vectorizer.get_feature_names_out()
print(f"\n   📋 Sample skills in vocabulary:")
for skill in feature_names[:10]:
    print(f"      • {skill}")

# ============================================
# STEP 4: USER INPUT (Cold Start Solution - Onboarding Survey)
# ============================================

print("\n" + "="*60)
print("🎯 STEP 4: User Profile Creation")
print("="*60)

print("\n📌 Cold Start Problem Solution: Onboarding Survey")
print("   Since you're a new user, please tell us about your skills!")
print("-"*40)

# Get 3 skills from user
print("\n❓ Please enter 3 skills you have or want to work with:")
print("   (Examples: Python, JavaScript, AWS, React, SQL, Docker, etc.)")
print()

skills_input = []
for i in range(3):
    skill = input(f"   Skill {i+1}: ").strip().lower()
    skills_input.append(skill)

user_skills_text = " ".join(skills_input)

print(f"\n   ✅ Your skills: {', '.join(skills_input)}")
print(f"   📌 Profile vector created with TF-IDF")

# ============================================
# STEP 5: TRANSFORM USER INPUT TO VECTOR
# ============================================

print("\n📊 STEP 5: Transforming User Profile to Vector")
print("-"*40)

# Transform user skills using the SAME vectorizer
user_vector = vectorizer.transform([user_skills_text])

print(f"   ✅ User vector created!")
print(f"   • Shape: {user_vector.shape}")

# Show which skills were recognized
user_feature_indices = user_vector.nonzero()[1]
recognized_skills = [feature_names[i] for i in user_feature_indices]

if recognized_skills:
    print(f"\n   📋 Skills recognized by the system:")
    for skill in recognized_skills:
        score = user_vector[0, user_vector.nonzero()[1][recognized_skills.index(skill)]]
        print(f"      • {skill}: {score:.3f} (TF-IDF weight)")
else:
    print("\n   ⚠️ No skills recognized! Try using more common tech terms.")
    print("      Examples: Python, Java, JavaScript, AWS, SQL")

# ============================================
# STEP 6: CALCULATE COSINE SIMILARITY
# ============================================

print("\n📊 STEP 6: Calculating Cosine Similarity")
print("-"*40)

# Calculate similarity between user and all roles
similarity_scores = cosine_similarity(user_vector, tfidf_matrix).flatten()

print("   ✅ Similarity scores calculated!")
print("   📌 Cosine Similarity measures the ANGLE between vectors")
print("      • Score = 1.0 → Perfect match (same direction)")
print("      • Score = 0.0 → No match (perpendicular)")

# ============================================
# STEP 7: SORT AND GET TOP-N RECOMMENDATIONS
# ============================================

print("\n📊 STEP 7: Sorting & Filtering (Top-N Recommendations)")
print("-"*40)

# Create results dataframe
results = pd.DataFrame({
    'role': df['role'],
    'skills': df['skills'],
    'similarity_score': similarity_scores
})

# Sort by similarity score (descending)
results = results.sort_values('similarity_score', ascending=False)

# Get Top 3 recommendations
top_n = 3
top_recommendations = results.head(top_n)

print(f"   ✅ Top {top_n} recommendations selected!")
print(f"   📌 Filtering prevents 'choice overload'")

# ============================================
# STEP 8: DISPLAY RECOMMENDATIONS
# ============================================

print("\n" + "="*60)
print("🎯 YOUR PERSONALIZED RECOMMENDATIONS")
print("="*60)

print(f"\n📌 Based on your skills: {', '.join(skills_input).upper()}")
print("\n" + "-"*60)

for idx, row in top_recommendations.iterrows():
    score = row['similarity_score'] * 100
    print(f"\n🏆 #{top_recommendations.index.get_loc(idx) + 1}: {row['role']}")
    print(f"   📊 Match Score: {score:.1f}%")
    print(f"   🔧 Required Skills: {row['skills']}")

    # Show which of user's skills match
    user_skills_lower = [s.lower() for s in skills_input]
    role_skills_lower = row['skills'].lower()
    matching = [s for s in user_skills_lower if s in role_skills_lower]
    if matching:
        print(f"   ✅ Your matching skills: {', '.join(matching)}")

print("\n" + "-"*60)

# Show low-score recommendations if any
if top_recommendations.iloc[0]['similarity_score'] < 0.1:
    print("\n⚠️ Note: Your similarity scores are low.")
    print("   Try using more specific tech skills (e.g., 'React' instead of 'frontend')")

# ============================================
# STEP 9: BONUS - EXPLORE OTHER ROLES (Optional)
# ============================================

print("\n" + "="*60)
print("📊 BONUS: Complete Ranking of All Roles")
print("="*60)

print("\n   All roles ranked by similarity to your profile:")
print("   " + "-"*50)

for idx, row in results.iterrows():
    score = row['similarity_score'] * 100
    bar = '█' * int(score/5) + '░' * (20 - int(score/5))
    print(f"   {row['role'][:25]:25} {bar} {score:5.1f}%")

# ============================================
# STEP 10: HANDLE COLD START - If no matches found
# ============================================

if top_recommendations.iloc[0]['similarity_score'] == 0:
    print("\n" + "!"*60)
    print("⚠️ COLD START DETECTED: No skills matched!")
    print("!"*60)
    print("\n📌 Since you're a new user with no matching skills,")
    print("   here are the most popular/trending tech roles:")
    print("\n   🔥 TRENDING ROLES (Fallback Recommendation):")

    trending = ['Data Scientist', 'Machine Learning Engineer', 'DevOps Engineer']
    for i, role in enumerate(trending, 1):
        print(f"      {i}. {role}")

    print("\n   💡 Tip: Re-run and try skills like:")
    print("      'Python', 'JavaScript', 'AWS', 'SQL', 'React'")

# ============================================
# FINAL SUMMARY
# ============================================

print("\n" + "="*60)
print("📊 PROJECT 3 SUMMARY REPORT")
print("="*60)

print("""
✅ COMPLETED REQUIREMENTS:
   ✓ Took 3 user inputs (skills/interests)
   ✓ Used Content-Based Filtering (not collaborative)
   ✓ Applied TF-IDF for feature weighting
   ✓ Used Cosine Similarity for matching
   ✓ Returned Top 3 recommendations
   ✓ Handled Cold Start with onboarding survey + trending fallback

🔧 TECHNICAL COMPONENTS:
   • Vector Space Model: TF-IDF transformation
   • Similarity Metric: Cosine Similarity
   • Dataset: 15 tech roles with skill requirements
   • Ranking: Descending similarity score
   • Filtering: Top-N (3 recommendations)
""")

print("="*60)
print("🎉 Congratulations! You've completed Project 3 at DecodeLabs!")
print("💡 Next step: Try adding more roles or customizing the dataset!")
print("="*60)

🎯 DecodeLabs - Project 3: AI Recommendation Logic
📌 Tech Stack Recommender (Content-Based Filtering)

📊 STEP 1: Loading Tech Roles Dataset
----------------------------------------
   ✅ Loaded 15 tech roles

   📋 Available Roles:
      1. Frontend Developer
      2. Backend Developer
      3. Full Stack Developer
      4. Data Scientist
      5. Machine Learning Engineer
      6. DevOps Engineer
      7. Cloud Architect
      8. Mobile Developer (iOS)
      9. Mobile Developer (Android)
      10. Cybersecurity Analyst
      11. Database Administrator
      12. QA Engineer
      13. Product Manager
      14. UI/UX Designer
      15. Game Developer

📊 STEP 2: Preparing Skill Data for TF-IDF
----------------------------------------
   Sample document (Backend Developer):
   → Python, Java, Node.js, SQL, REST APIs, Spring Boot, Django

   📌 TF-IDF will:
      • Penalize common words (like 'Python' if too frequent)
      • Reward unique/specific skills

📊 STEP 3: Applying TF-IDF Vectorizatio